# Final Fire and Smoke Evaluation

This notebook compares the final detector and crop verifier with the preceding model pair on validation images and 12 labeled evaluation videos. No models are trained here.

Reruns use the third fine-tuning and threshold-tuning notebook state in the same kernel, plus the named video inputs. The saved results can be reviewed without running the notebook.


## 01. Video evaluation helpers

Defines the sequential video runner and temporal alarm rules. Both model pairs use detector confidence 0.20, verifier probability 0.70, and a target rate of 5 FPS.


In [ ]:
# Helpers from the completed video evaluation. Requires FT3 notebook state.
import math
from collections import Counter, deque
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import torch
import torchvision
from IPython.display import display
from tqdm.auto import tqdm

DET_CONF, VER_CONF, TARGET_FPS = 0.20, 0.70, 5
RULES = {"fire": (1.0, 0.8, 0.5, 0.8, 1.0), "smoke": (2.0, 0.6, 0.4, 0.7, 1.5)}
VIDEO_PATH = None

def alarm_tracker():
    tracks, active, last_hit = [], {}, {}

    def center(box):
        return (box[:2] + box[2:]) / 2

    def size(box):
        return max(box[2] - box[0], box[3] - box[1], 1.0)

    def step(t, detections, sample_fps):
        alarms = []
        for label, (window, ratio, normal, moving, move_cut) in RULES.items():
            ds = [(box, score) for cls, box, score in detections if cls == label]
            if ds:
                last_hit[label] = t
            if active.get(label) and t - last_hit[label] > 10.0:
                active[label] = False

            candidates = []
            for i, track in enumerate(tracks):
                if track["label"] != label:
                    continue
                a = track["box"]
                for j, (b, _) in enumerate(ds):
                    overlap = box_iou([a], [b])[0, 0]
                    dist = np.linalg.norm(center(a) - center(b)) / size(a)
                    if overlap >= 0.1 or dist <= 1.0:
                        score = 1 + overlap if overlap > 0 else 1 - min(dist, 1)
                        candidates.append((score, i, j))

            used_tracks, used_dets = set(), set()

            def add_hit(track, j):
                box, score = ds[j]
                track.update(box=box, last=t)
                track["hits"].append((t, score, center(box), size(box)))

            for _, i, j in sorted(candidates, reverse=True):
                if i not in used_tracks and j not in used_dets:
                    add_hit(tracks[i], j)
                    used_tracks.add(i)
                    used_dets.add(j)

            for j in range(len(ds)):
                if j not in used_dets:
                    track = {"label": label, "hits": deque()}
                    add_hit(track, j)
                    tracks.append(track)

            required = max(1, math.ceil(ratio * window * max(0.5, sample_fps)))
            for track in tracks:
                if track["label"] != label or track["last"] != t:
                    continue
                hits = track["hits"]
                while hits and hits[0][0] < t - window:
                    hits.popleft()
                if len(hits) < required or active.get(label):
                    continue
                movement = np.linalg.norm(hits[-1][2] - hits[0][2]) / max(
                    np.mean([hit[3] for hit in hits]), 1.0
                )
                score = float(np.mean([hit[1] for hit in hits]))
                if score >= (moving if movement > move_cut else normal):
                    active[label] = True
                    alarms.append((label, t))

        tracks[:] = [track for track in tracks if t - track["last"] <= 2.0]
        return alarms

    return step

def run_fs(name, detector_path, verifier_path):
    for path in (detector_path, verifier_path):
        if not Path(path).is_file():
            raise FileNotFoundError(path)

    device = f"cuda:{DEVICE}" if torch.cuda.is_available() else "cpu"
    det = detector_for(detector_path)
    ck = torch.load(verifier_path, map_location="cpu", weights_only=True)
    net = torchvision.models.efficientnet_b0(weights=None)
    net.classifier[1] = torch.nn.Linear(
        net.classifier[1].in_features, len(ck["classes"])
    )
    net.load_state_dict(ck["state_dict"])
    net.to(device).eval()
    event_idx = [ck["classes"].index(c) for c in ("fire", "smoke")]
    side, pad = int(ck.get("imgsz", 224)), float(ck.get("pad", 0.3))
    del ck

    mean = np.array([.485, .456, .406], dtype=np.float32)
    std = np.array([.229, .224, .225], dtype=np.float32)
    options = {"quantize": 16} if device.startswith("cuda") else {}
    cap = cv2.VideoCapture(str(VIDEO_PATH))
    tally, events = Counter(), []
    update = alarm_tracker()

    try:
        if not cap.isOpened():
            raise RuntimeError(f"Cannot open video: {VIDEO_PATH}")
        fps = cap.get(cv2.CAP_PROP_FPS)
        if not 1.0 <= fps <= 240.0:
            raise RuntimeError(f"Invalid video FPS: {fps}")
        stride = max(1, round(fps / TARGET_FPS))
        expected = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        with tqdm(total=expected or None, desc=name, mininterval=2) as bar:
            while True:
                ok, frame = cap.read()
                if not ok:
                    break
                i = tally["read"]
                tally["read"] += 1
                bar.update(1)
                if i % stride:
                    continue

                tally["processed"] += 1
                result = det.predict(
                    frame, imgsz=800, conf=DET_CONF, iou=.50,
                    max_det=50, device=device, verbose=False, **options
                )[0]
                items = pred_boxes(result)
                tally["yolo_flagged_frames"] += bool(items)

                crops, meta = [], []
                for cls, score, box in items:
                    crop = square_crop(frame, box, pad=pad)
                    if crop is None:
                        continue
                    interp = cv2.INTER_AREA if crop.shape[0] > side else cv2.INTER_LINEAR
                    crop = cv2.resize(crop, (side, side), interpolation=interp)
                    rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB).astype(np.float32) / 255
                    crops.append((rgb - mean) / std)
                    meta.append((NAMES[cls], box, score))

                verified = []
                if crops:
                    x = np.ascontiguousarray(np.stack(crops).transpose(0, 3, 1, 2))
                    with torch.inference_mode():
                        probs = net(torch.from_numpy(x).to(device)).float().softmax(1)
                        scores = probs.cpu().numpy()[:, event_idx].sum(axis=1)
                    verified = [
                        (label, box, math.sqrt(score * float(p)))
                        for (label, box, score), p in zip(meta, scores)
                        if float(p) >= VER_CONF
                    ]

                tally["verified_flagged_frames"] += bool(verified)
                events.extend(update(i / fps, verified, fps / stride))

        if not tally["processed"] or (expected and tally["read"] < expected):
            raise RuntimeError(f"Incomplete video read: {tally['read']}/{expected} frames.")
    finally:
        cap.release()
        del det, net
        release_gpu()

    counts = Counter(label for label, _ in events)
    return {
        "pair": name,
        "frames_checked": tally["processed"],
        "yolo_flagged_frames": tally["yolo_flagged_frames"],
        "verified_flagged_frames": tally["verified_flagged_frames"],
        "false_fire_alarms": counts["fire"],
        "false_smoke_alarms": counts["smoke"],
        "total_false_alarms": len(events),
        "alarm_times_s": ", ".join(f"{label} {t:.2f}" for label, t in events),
    }


## 02. Score the final pair on images

Reports overall and per-class precision, recall, F1, and AP at several IoU thresholds for standard and hard validation.


In [31]:
import numpy as np
import pandas as pd
from IPython.display import display

FINAL_CUTOFF = (0.20, 0.70)
IOU_LEVELS = np.linspace(0.50, 0.95, 10)

required = ("tune_val", "tune_hard", "tune_score", "TUNE_OUT")
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f"Missing notebook state: {missing}. Send me this error. Don't run all.")

def ap101(tp, n_gt):
    if n_gt == 0:
        return np.nan
    if len(tp) == 0:
        return 0.0
    hits = np.cumsum(tp)
    recall = hits / n_gt
    precision = hits / np.arange(1, len(tp) + 1)
    envelope = np.maximum.accumulate(precision[::-1])[::-1]
    idx = np.searchsorted(recall, np.linspace(0, 1, 101), side="left")
    return float(np.r_[envelope, 0.0][idx].mean())

def final_ap(cache):
    d_min, v_min = FINAL_CUTOFF
    by_class = []

    for cls in (0, 1):
        n_gt = 0
        predictions = []

        for truth, pred, det, ver, iou in cache:
            gt_idx = np.flatnonzero(np.asarray(truth) == cls)
            det_idx = np.flatnonzero(
                (np.asarray(pred) == cls) & (det > d_min) & (ver >= v_min)
            )
            n_gt += len(gt_idx)
            if not len(det_idx):
                continue

            # Use the same combined confidence as the alarm pipeline.
            scores = np.sqrt(det[det_idx] * ver[det_idx])
            order = np.argsort(-scores, kind="stable")
            scores = scores[order]
            overlaps = iou[np.ix_(gt_idx, det_idx[order])].T
            tp = np.zeros((len(scores), len(IOU_LEVELS)), dtype=bool)

            for k, threshold in enumerate(IOU_LEVELS):
                available = np.ones(len(gt_idx), dtype=bool)
                for j, overlap in enumerate(overlaps):
                    candidates = np.flatnonzero(available & (overlap >= threshold))
                    if len(candidates):
                        match = candidates[np.argmax(overlap[candidates])]
                        available[match] = False
                        tp[j, k] = True

            predictions.extend(zip(scores, tp))

        predictions.sort(key=lambda item: -item[0])
        hits = (
            np.stack([tp for _, tp in predictions])
            if predictions else np.empty((0, len(IOU_LEVELS)), dtype=bool)
        )
        by_class.append([ap101(hits[:, k], n_gt) for k in range(len(IOU_LEVELS))])

    return np.asarray(by_class)

rows = []
for split, cache in (("val", tune_val), ("hard_val", tune_hard)):
    scores, _ = tune_score(cache, FINAL_CUTOFF)
    ap = final_ap(cache)

    for idx, name in enumerate(("all", "fire", "smoke")):
        if name == "all":
            p, r, f1 = scores["precision"], scores["recall"], scores["F1"]
            a = np.nanmean(ap, axis=0)
        else:
            p, r, f1 = (scores[f"{name}_{key}"] for key in ("P", "R", "F1"))
            a = ap[idx - 1]

        rows.append({
            "split": split,
            "class": name,
            "images": len(cache),
            "precision_%": 100 * p,
            "recall_%": 100 * r,
            "F1_%": 100 * f1,
            "AP50_%": 100 * a[0],
            "AP95_%": 100 * a[-1],
            "AP50-95_%": 100 * a.mean(),
        })

final_eval = pd.DataFrame(rows)
display(final_eval.round(2))

out = TUNE_OUT / "final_model_evaluation.csv"
final_eval.to_csv(out, index=False)
print("Final settings: YOLO 0.20 | verifier 0.70")
print("Saved:", out)

,split,class,images,precision_%,recall_%,F1_%,AP50_%,AP95_%,AP50-95_%
0,val,all,3764,74.96,56.87,64.67,52.66,2.40,33.88
1,val,fire,3764,69.17,55.62,61.66,49.38,1.63,29.07
2,val,smoke,3764,84.92,58.72,69.43,55.94,3.17,38.70
3,hard_val,all,869,61.63,64.38,62.98,53.95,0.80,29.09
4,hard_val,fire,869,58.54,74.42,65.53,60.64,0.04,33.09
5,hard_val,smoke,869,65.96,55.16,60.08,47.26,1.56,25.09


Final settings: YOLO 0.20 | verifier 0.70
Saved: /kaggle/working/final_fire_smoke/20261115_102627_437619/last_epoch_verifier/threshold_tuning/final_model_evaluation.csv


## 03. Compare the two model pairs

Evaluates the second fine-tuning run with the original verifier and the third run with the final verifier on the same images. The comparison includes flagged background images.


In [32]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

for path in (base_det, base_ver):
    if not Path(path).is_file():
        raise FileNotFoundError(f"Missing old checkpoint: {path}")

print("Old detector:", base_det)
print("Old verifier:", base_ver)

# Temporarily point the existing evaluator at the old pair.
saved_yolo, saved_ver = TUNE_YOLO, TUNE_VER
try:
    TUNE_YOLO, TUNE_VER = base_det, base_ver
    old_val = tune_collect("val")
    old_hard = tune_collect("hard_val")
finally:
    TUNE_YOLO, TUNE_VER = saved_yolo, saved_ver

rows = []
for split, old_cache, new_cache in (
    ("val", old_val, tune_val),
    ("hard_val", old_hard, tune_hard),
):
    for name, cache in (
        ("2nd fine-tune YOLO + old verifier", old_cache),
        ("Epoch 11 YOLO + final verifier", new_cache),
    ):
        scores, _ = tune_score(cache, FINAL_CUTOFF)
        ap = np.nanmean(final_ap(cache), axis=0)
        rows.append({
            "split": split,
            "model_pair": name,
            "images": len(cache),
            "precision_%": 100 * scores["precision"],
            "recall_%": 100 * scores["recall"],
            "F1_%": 100 * scores["F1"],
            "mAP50_%": 100 * ap[0],
            "mAP50-95_%": 100 * ap.mean(),
            "clean_images_flagged": scores["bg_images_flagged"],
        })

pair_comparison = pd.DataFrame(rows)
display(pair_comparison.round(2))

out = TUNE_OUT / "old_vs_final_pair_evaluation.csv"
pair_comparison.to_csv(out, index=False)
print("Saved:", out)

Old detector: /tmp/fire_smoke_final_20261115_102627_437619/detector_v4_kaggle.pt
Old verifier: /tmp/fire_smoke_final_20261115_102627_437619/verifier.pt


Scoring val:   0%|          | 0/3764 [00:00<?, ?it/s]

Scoring hard_val:   0%|          | 0/869 [00:00<?, ?it/s]

,split,model_pair,images,precision_%,recall_%,F1_%,mAP50_%,mAP50-95_%,clean_images_flagged
0,val,2nd fine-tune YOLO + old verifier,3764,77.30,56.85,65.51,55.56,36.72,39
1,val,Epoch 11 YOLO + final verifier,3764,74.96,56.87,64.67,52.66,33.88,72
2,hard_val,2nd fine-tune YOLO + old verifier,869,71.26,67.16,69.15,59.53,32.90,31
3,hard_val,Epoch 11 YOLO + final verifier,869,61.63,64.38,62.98,53.95,29.09,49


Saved: /kaggle/working/final_fire_smoke/20261115_102627_437619/last_epoch_verifier/threshold_tuning/old_vs_final_pair_evaluation.csv


## 04. Evaluate six fire and smoke clips

Checks for the expected alarm on three fire clips and three smoke clips, and records the time of the first matching alarm from clip start.


In [34]:
from pathlib import Path
import re
import pandas as pd
from IPython.display import display

clips = {
    "f1.mp4": "fire", "f2.mp4": "fire", "f3.mp4": "fire",
    "s1.mp4": "smoke", "s2.mp4": "smoke", "s3.mp4": "smoke",
}
input_files = [p for p in Path("/kaggle/input").rglob("*") if p.is_file()]

clip_paths = {}
for name in clips:
    matches = [p for p in input_files if p.name.lower() == name.lower()]
    if len(matches) != 1:
        raise RuntimeError(f"Expected exactly one {name}. Found: {matches}")
    clip_paths[name] = matches[0]

positive_pairs = [
    ("2nd fine-tune + old verifier", base_det, base_ver),
    ("3rd fine-tune + final verifier", NEW_DET, NEW_VER),
]

for _, detector_path, verifier_path in positive_pairs:
    for path in (detector_path, verifier_path):
        if not Path(path).is_file():
            raise FileNotFoundError(path)

assert (DET_CONF, VER_CONF, TARGET_FPS) == (0.20, 0.70, 5), (
    "Expected the same settings as the negative clips: 0.20 / 0.70, target 5 FPS."
)

positive_rows = []
previous_video = VIDEO_PATH

try:
    for name, expected in clips.items():
        VIDEO_PATH = clip_paths[name]
        print(f"\nTesting {name} | expected alarm: {expected}")

        for pair in positive_pairs:
            result = run_fs(*pair)
            # run_fs uses "false_*" column names from the negative-clip test.
            # On these clips, count fire and smoke alarms without calling them false.
            fire_alarms = result["false_fire_alarms"]
            smoke_alarms = result["false_smoke_alarms"]
            match = re.search(
                rf"\b{expected}\s+(\d+(?:\.\d+)?)",
                result["alarm_times_s"],
            )

            positive_rows.append({
                "video": name,
                "expected": expected,
                "pair": result["pair"],
                "frames_checked": result["frames_checked"],
                "yolo_flagged_frames": result["yolo_flagged_frames"],
                "verified_flagged_frames": result["verified_flagged_frames"],
                "fire_alarms": fire_alarms,
                "smoke_alarms": smoke_alarms,
                "expected_alarm_detected": (fire_alarms if expected == "fire"
                                            else smoke_alarms) > 0,
                "first_expected_alarm_s": float(match.group(1)) if match else None,
                "alarm_times_s": result["alarm_times_s"],
            })
finally:
    VIDEO_PATH = previous_video

positive_report = pd.DataFrame(positive_rows)
assert positive_report.groupby("video")["frames_checked"].nunique().eq(1).all(), (
    "The two pairs did not check the same number of frames."
)

positive_summary = (
    positive_report.groupby(["pair", "expected"], sort=False)
    .agg(
        clips_tested=("video", "size"),
        clips_detected=("expected_alarm_detected", "sum"),
    )
    .reset_index()
)

print("\nPer-video results")
display(positive_report)

print("\nExpected alarms caught by each pair")
display(positive_summary)

positive_report.to_csv(RUN / "six_positive_clips_per_video.csv", index=False)
positive_summary.to_csv(RUN / "six_positive_clips_summary.csv", index=False)
print("Saved both CSVs in", RUN)


Testing f1.mp4 | expected alarm: fire


2nd fine-tune + old verifier:   0%|          | 0/7080 [00:00<?, ?it/s]

3rd fine-tune + final verifier:   0%|          | 0/7080 [00:00<?, ?it/s]


Testing f2.mp4 | expected alarm: fire


2nd fine-tune + old verifier:   0%|          | 0/4230 [00:00<?, ?it/s]

3rd fine-tune + final verifier:   0%|          | 0/4230 [00:00<?, ?it/s]

... (13 outputs left out)

,video,expected,pair,frames_checked,yolo_flagged_frames,verified_flagged_frames,fire_alarms,smoke_alarms,expected_alarm_detected,first_expected_alarm_s,alarm_times_s
0,f1.mp4,fire,2nd fine-tune + old verifier,1180,1177,1171,1,1,True,0.60,"fire 0.60, smoke 212.00"
1,f1.mp4,fire,3rd fine-tune + final verifier,1180,1180,1180,1,0,True,0.60,fire 0.60
2,f2.mp4,fire,2nd fine-tune + old verifier,353,352,304,1,0,True,0.60,fire 0.60
3,f2.mp4,fire,3rd fine-tune + final verifier,353,327,326,1,0,True,0.60,fire 0.60
4,f3.mp4,fire,2nd fine-tune + old verifier,40,40,40,1,0,True,0.60,fire 0.60
5,f3.mp4,fire,3rd fine-tune + final verifier,40,40,40,1,0,True,0.60,fire 0.60
6,s1.mp4,smoke,2nd fine-tune + old verifier,143,142,142,0,1,True,1.04,smoke 1.04
7,s1.mp4,smoke,3rd fine-tune + final verifier,143,134,5,0,0,False,NaN,
8,s2.mp4,smoke,2nd fine-tune + old verifier,495,467,431,1,0,False,NaN,fire 1.67
9,s2.mp4,smoke,3rd fine-tune + final verifier,495,470,372,1,0,False,NaN,fire 0.63



Expected alarms caught by each pair


,pair,expected,clips_tested,clips_detected
0,2nd fine-tune + old verifier,fire,3,3
1,3rd fine-tune + final verifier,fire,3,3
2,2nd fine-tune + old verifier,smoke,3,2
3,3rd fine-tune + final verifier,smoke,3,1


Saved both CSVs in /kaggle/working/final_fire_smoke/20261115_102627_437619/last_epoch_verifier


## 05. Evaluate six traffic clips

Counts verified detections and false alarms on six clips without fire or smoke, using the same settings for both model pairs.


In [35]:
from pathlib import Path

import pandas as pd
from IPython.display import display

clip_names = [f"r{i}.mp4" for i in range(1, 7)]
input_files = [
    p for p in Path("/kaggle/input").rglob("*")
    if p.is_file() and p.suffix.lower() == ".mp4"
]

traffic_paths = {}
for name in clip_names:
    matches = [p for p in input_files if p.name.lower() == name]
    if len(matches) != 1:
        raise RuntimeError(f"Expected one file named {name}. Found: {matches}")
    traffic_paths[name] = matches[0]

traffic_pairs = [
    ("2nd fine-tune + old verifier", base_det, base_ver),
    ("3rd fine-tune + final verifier", NEW_DET, NEW_VER),
]

for _, detector_path, verifier_path in traffic_pairs:
    for path in (detector_path, verifier_path):
        if not Path(path).is_file():
            raise FileNotFoundError(path)

if (DET_CONF, VER_CONF, TARGET_FPS) != (0.20, 0.70, 5):
    raise RuntimeError("Expected YOLO 0.20, verifier 0.70, and target 5 FPS.")

traffic_rows = []
previous_video = VIDEO_PATH

try:
    for name, path in traffic_paths.items():
        VIDEO_PATH = path
        print(f"\nTesting {name}")
        for pair in traffic_pairs:
            traffic_rows.append({"video": name, **run_fs(*pair)})
finally:
    VIDEO_PATH = previous_video

traffic_report = pd.DataFrame(traffic_rows)
if not traffic_report.groupby("video")["frames_checked"].nunique().eq(1).all():
    raise RuntimeError("The model pairs did not check the same number of frames.")

count_cols = [
    "frames_checked",
    "yolo_flagged_frames",
    "verified_flagged_frames",
    "false_fire_alarms",
    "false_smoke_alarms",
    "total_false_alarms",
]
traffic_totals = (
    traffic_report.groupby("pair", sort=False)[count_cols].sum().reset_index()
)
traffic_totals["verified_flagged_%"] = (
    100 * traffic_totals["verified_flagged_frames"] / traffic_totals["frames_checked"]
)

print("\nPer-video results")
display(traffic_report[["video", "pair", *count_cols, "alarm_times_s"]])

print("\nTotals across the six traffic clips")
display(traffic_totals.round(2))

traffic_report.to_csv(RUN / "six_traffic_clips_per_video.csv", index=False)
traffic_totals.to_csv(RUN / "six_traffic_clips_totals.csv", index=False)
print("Saved both CSVs in", RUN)


Testing r1.mp4


2nd fine-tune + old verifier:   0%|          | 0/561 [00:00<?, ?it/s]

3rd fine-tune + final verifier:   0%|          | 0/561 [00:00<?, ?it/s]


Testing r2.mp4


2nd fine-tune + old verifier:   0%|          | 0/3093 [00:00<?, ?it/s]

3rd fine-tune + final verifier:   0%|          | 0/3093 [00:00<?, ?it/s]

... (13 outputs left out)

,video,pair,frames_checked,yolo_flagged_frames,verified_flagged_frames,false_fire_alarms,false_smoke_alarms,total_false_alarms,alarm_times_s
0,r1.mp4,2nd fine-tune + old verifier,94,0,0,0,0,0,
1,r1.mp4,3rd fine-tune + final verifier,94,0,0,0,0,0,
2,r2.mp4,2nd fine-tune + old verifier,258,2,1,0,0,0,
3,r2.mp4,3rd fine-tune + final verifier,258,3,3,0,0,0,
4,r3.mp4,2nd fine-tune + old verifier,68,0,0,0,0,0,
5,r3.mp4,3rd fine-tune + final verifier,68,0,0,0,0,0,
6,r4.mp4,2nd fine-tune + old verifier,93,0,0,0,0,0,
7,r4.mp4,3rd fine-tune + final verifier,93,0,0,0,0,0,
8,r5.mp4,2nd fine-tune + old verifier,43,1,1,0,0,0,
9,r5.mp4,3rd fine-tune + final verifier,43,0,0,0,0,0,



Totals across the six traffic clips


,pair,frames_checked,yolo_flagged_frames,verified_flagged_frames,false_fire_alarms,false_smoke_alarms,total_false_alarms,verified_flagged_%
0,2nd fine-tune + old verifier,711,3,2,0,0,0,0.28
1,3rd fine-tune + final verifier,711,3,3,0,0,0,0.42


Saved both CSVs in /kaggle/working/final_fire_smoke/20261115_102627_437619/last_epoch_verifier
